* 최신 vllm 버전 요구됨

In [17]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from datasets import Dataset

import pandas as pd
import numpy as np

import torch
import json

/root/anaconda3/envs/vLLM/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def build_system_content(original_text):
    return (
        "You are an expert medical evaluator.\n\n"
        "Below is the ORIGINAL discharge summary of a patient.\n"
        "This is the ground-truth clinical record.\n\n"
        "===== ORIGINAL DISCHARGE SUMMARY =====\n"
        f"{original_text}\n"
        "======================================\n\n"
        "You will be given 5 generated clinical summaries of the SAME patient.\n"
        "Evaluate each summary independently, but assign scores based on RELATIVE quality,\n"
        "following the priority order specified below.\n\n"
        "EVALUATION PRIORITIES (VERY IMPORTANT):\n\n"
        "Priority 1 (MOST IMPORTANT): Correct use of REQUIRED TEST NAMES and SEX\n"
        "- The following test names must be explicitly and correctly written:\n"
        "  - Vitals: BP, HR, RR, Temp\n"
        "  - Labs: WBC, RBC, Hgb, Hct, Plt\n"
        "  - Red cell indices and others: MCV, MCH, MCHC, RDW, Glucose\n"
        "- Patient sex (Male/Female) must be correctly stated\n"
        "- Incorrect, missing, or improperly named tests or sex are penalized\n"
        "- This priority ALWAYS outweighs all lower priorities\n\n"
        "Priority 2: Correctness of VALUES corresponding to each test name\n"
        "- When a value is explicitly stated, it must match the discharge summary exactly\n"
        "- If a test is NOT present in the discharge summary, NA / not available / not reported\n"
        "  is considered CORRECT\n"
        "- Hallucinated, incorrect, or contradictory values are penalized\n\n"
        "Priority 3: Appropriateness of the clinical finding sentence\n"
        "- The sentence following \"The most diagnostic relevant finding was ...\" should:\n"
        "  - Describe a clinically meaningful finding or observation\n"
        "  - NOT be merely a diagnosis name\n"
        "  - Be consistent with the discharge summary\n\n"
        "SCORING RULES:\n"
        "- Scores represent RELATIVE ranks among the 5 summaries (5 = worst, 1 = best)\n"
        "- Multiple summaries may share the same score if their quality is comparable\n"
        "- Even if all summaries are poor, you MUST still rank them relative to each other\n\n"
        "OUTPUT RULES (STRICT):\n"
        "- You MUST output ONLY a single-line valid JSON object\n"
        "- The JSON must map summary index to score\n"
        "- Any token outside the JSON object is STRICTLY FORBIDDEN\n"
        "- Do NOT provide explanations, reasoning, comments, or natural language\n"
        "- Do NOT use markdown or headings\n\n"
        "Example:\n"
        "{\"1\": 1, \"2\": 3, \"3\": 2, \"4\": 4, \"5\": 5}"
    )

In [1]:
model_name = "Qwen/Qwen3.5-35B-A3B"

In [ ]:
llm = LLM(
    model = model_name,
    dtype = torch.bfloat16,
    trust_remote_code = True,
    max_model_len = 32768,
    gpu_memory_utilization = 0.9
)

INFO 04-07 01:54:44 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': torch.bfloat16, 'max_model_len': 32768, 'disable_log_stats': True, 'model': 'Qwen/Qwen3.5-35B-A3B'}
INFO 04-07 01:54:46 [model.py:549] Resolved architecture: Qwen3_5MoeForConditionalGeneration
INFO 04-07 01:54:46 [model.py:1678] Using max model len 32768


2026-04-07 01:54:46,692	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 04-07 01:54:46 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.


`Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.


INFO 04-07 01:54:48 [config.py:281] Setting attention block size to 1056 tokens to ensure that attention page size is >= mamba page size.
INFO 04-07 01:54:48 [config.py:312] Padding mamba page size by 0.76% to ensure that mamba page size and attention page size are exactly equal.
INFO 04-07 01:54:48 [vllm.py:790] Asynchronous scheduling is enabled.


...

(EngineCore pid=198679) INFO 04-07 01:55:29 [gpu_model_runner.py:4735] Starting to load model Qwen/Qwen3.5-35B-A3B...
(EngineCore pid=198679) INFO 04-07 01:55:30 [cuda.py:390] Using backend AttentionBackendEnum.FLASH_ATTN for vit attention
(EngineCore pid=198679) INFO 04-07 01:55:30 [mm_encoder_attention.py:230] Using AttentionBackendEnum.FLASH_ATTN for MMEncoderAttention.
(EngineCore pid=198679) INFO 04-07 01:55:30 [gdn_linear_attn.py:139] Using FlashInfer GDN prefill kernel
(EngineCore pid=198679) INFO 04-07 01:55:30 [gdn_linear_attn.py:140] FlashInfer GDN prefill kernel is JIT-compiled; first run may take a while to compile. Set `--gdn-prefill-backend triton` to avoid JIT compile time.
(EngineCore pid=198679) INFO 04-07 01:55:30 [unquantized.py:186] Using TRITON backend for Unquantized MoE
(EngineCore pid=198679) INFO 04-07 01:55:30 [cuda.py:334] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
(EngineCore p

(EngineCore pid=198679) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=198679) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/14 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:   7% Completed | 1/14 [00:00<00:07,  1.75it/s]
Loading safetensors checkpoint shards:  14% Completed | 2/14 [00:01<00:07,  1.51it/s]
Loading safetensors checkpoint shards:  21% Completed | 3/14 [00:01<00:07,  1.49it/s]
Loading safetensors checkpoint shards:  29% Completed | 4/14 [00:02<00:06,  1.52it/s]
Loading safetensors checkpoint shards:  36% Completed | 5/14 [00:03<00:05,  1.57it/s]
Loading safetensors checkpoint shards:  43% C

(EngineCore pid=198679) INFO 04-07 01:55:39 [default_loader.py:384] Loading weights took 7.78 seconds
(EngineCore pid=198679) INFO 04-07 01:55:40 [gpu_model_runner.py:4820] Model loading took 65.53 GiB memory and 9.297086 seconds
(EngineCore pid=198679) INFO 04-07 01:55:40 [gpu_model_runner.py:5753] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 image items of the maximum feature size.
(EngineCore pid=198679) INFO 04-07 01:55:46 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/a5d091b9d3/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=198679) INFO 04-07 01:55:46 [backends.py:1111] Dynamo bytecode transform time: 5.29 s
(EngineCore pid=198679) INFO 04-07 01:55:47 [backends.py:372] Cache the graph of compile range (1, 16384) for later use
(EngineCore pid=198679) INFO 04-07 01:56:04 [backends.py:390] Compiling a graph for compile range (1, 16384) takes 17.71 s
(EngineCore pid=198679) INFO 04-07 01:56:06 [decor

(EngineCore pid=198679) 2026-04-07 01:56:13,753 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=198679) 2026-04-07 01:56:13,787 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 10.79it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 51/51 [00:04<00:00, 11.26it/s]


(EngineCore pid=198679) INFO 04-07 01:56:23 [gpu_model_runner.py:6046] Graph capturing finished in 10 secs, took 1.83 GiB
(EngineCore pid=198679) INFO 04-07 01:56:23 [gpu_worker.py:597] CUDA graph pool memory: 1.83 GiB (actual), 1.53 GiB (estimated), difference: 0.3 GiB (16.5%).
(EngineCore pid=198679) INFO 04-07 01:56:23 [core.py:283] init engine (profile, create kv cache, warmup model) took 43.86 seconds
(EngineCore pid=198679) INFO 04-07 01:56:24 [vllm.py:790] Asynchronous scheduling is enabled.


In [ ]:
df_gen = pd.read_csv("data/generated_data_v1.1.2.csv")
df_discharge = pd.read_csv("data/gen_data_for_dpo_20260221.csv")

In [26]:
(df_gen.subject_id.unique() == df_discharge.subject_id.unique()).mean()

np.float64(1.0)

In [ ]:
summarized_text = df_gen.pivot_table(index = "subject_id", values = "generated_text",
                                     aggfunc = (lambda x: "\n\n".join([f"[{i+1}{"st" if i == 0 else "nd" if i == 1 else "rd" if i == 2 else "th"} generated text]\n\"\"\"\n{t}\n\"\"\"" for i, t in enumerate(x)])))\
                                        .reset_index()
df_gen = df_gen.assign(gen_num = [(i%5)+1 for i in range(df_gen.shape[0])])
df_wide = df_gen.pivot(index = "subject_id", columns = "gen_num", values = "generated_text").reset_index()
full_text = pd.merge(summarized_text, df_discharge[["subject_id", "text"]]).assign(system = lambda _df: _df.text.map(build_system_content)).drop(["text"], axis = 1)

In [72]:
ds = Dataset.from_pandas(full_text)
columns_to_remove = [f for f in list(ds.features) if f not in "subject_id"]

In [73]:
ds = ds.map(
    lambda sample:
    {"messages": [
        {"role": "system", "content": sample["system"]},
        {"role": "user", "content": sample["generated_text"]}
    ]}
)

Map: 100%|██████████| 2000/2000 [00:00<00:00, 10632.45 examples/s]


In [74]:
ds = ds.map(remove_columns = columns_to_remove, batched = False)

Map: 100%|██████████| 2000/2000 [00:00<00:00, 18333.43 examples/s]


In [ ]:
sampling_params = SamplingParams(temperature = 0.0, max_tokens = 64)
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast = True)

def template_dataset(example):
    return {"prompt": tokenizer.apply_chat_template(example["messages"], tokenize = False, add_generation_prompt = True)}

inference_data = ds.map(template_dataset, remove_columns = ["messages"])
prompts = inference_data["prompt"]

In [ ]:
outputs = llm.generate(prompts, sampling_params)

data = []
idx = inference_data["subject_id"]

for i, output in enumerate(outputs):
    current_subject_id = idx[i]

    try:
        label = json.loads(output.text.strip())
    except:
        label = pd.NA

    row = {
        "subject_id": current_subject_id,
        "label": label
    }
    
    data.append(row)

df = pd.DataFrame(data)

In [24]:
np.argmax(list(json.loads("{\"1\": 1, \"2\": 3, \"3\": 2, \"4\": 4, \"5\": 5}").values()))
np.argmin(list(json.loads("{\"1\": 1, \"2\": 3, \"3\": 2, \"4\": 4, \"5\": 5}").values()))

np.int64(0)

In [ ]:
data = []

for id, label in df[["subject_id", "label"]]:
    try:
        ## 겹치는 것이 있다면 가장 먼저를 선택
        max_idx = np.argmax(list(label)) + 1
        min_idx = np.argmin(list(label)) + 1
    except:
        continue

    row = {
        "subject_id": id,
        "text": df_discharge.loc[df_discharge.subject_id == id, "text"],
        "chosen": df_wide.loc[df_wide.subject_id == id, max_idx],
        "rejected": df_wide.loc[df_wide.subject_id == id, min_idx]
    }

    data.append(row)

dpo_dataset = pd.DaraFrame(data)
dpo_dataset.to_csv("data/dpo_dataset.csv", index = False, encoding = "utf-8-sig")